# EDA: raw pulled data

Loads the raw JSON from `data/raw/` (Open Targets + ClinicalTrials.gov) into DataFrames for exploration. No analysis here — just loading.

In [1]:
import json

import pandas as pd

from pharma_graphrag.config import DATA_RAW_DIR

## Open Targets: targets and drugs

In [2]:
target_rows = []
drug_rows = []

for path in sorted(DATA_RAW_DIR.glob("open_targets_*.json")):
    disease = json.loads(path.read_text())["data"]["disease"]
    disease_name = disease["name"]
    disease_id = disease["id"]

    for row in disease["associatedTargets"]["rows"]:
        target_rows.append({
            "disease_name": disease_name,
            "disease_id": disease_id,
            "score": row["score"],
            "target_id": row["target"]["id"],
            "target_symbol": row["target"]["approvedSymbol"],
        })

    for row in disease["drugAndClinicalCandidates"]["rows"]:
        drug_rows.append({
            "disease_name": disease_name,
            "disease_id": disease_id,
            "drug_id": row["drug"]["id"],
            "drug_name": row["drug"]["name"],
            "drug_type": row["drug"]["drugType"],
            "max_clinical_stage": row["maxClinicalStage"],
        })

targets_df = pd.DataFrame(target_rows)
drugs_df = pd.DataFrame(drug_rows)
targets_df.head()

,disease_name,disease_id,score,target_id,target_symbol
0,Crohn disease,MONDO_0005011,0.750739,ENSG00000167207,NOD2
1,Crohn disease,MONDO_0005011,0.743305,ENSG00000113302,IL12B
2,Crohn disease,MONDO_0005011,0.722699,ENSG00000115232,ITGA4
3,Crohn disease,MONDO_0005011,0.685287,ENSG00000096968,JAK2
4,Crohn disease,MONDO_0005011,0.681308,ENSG00000105397,TYK2


In [3]:
drugs_df.head()

,disease_name,disease_id,drug_id,drug_name,drug_type,max_clinical_stage
0,Crohn disease,MONDO_0005011,CHEMBL5314733,DUVAKITUG,Antibody,PHASE_3
1,Crohn disease,MONDO_0005011,CHEMBL1535,HYDROXYCHLOROQUINE,Small molecule,PHASE_2
2,Crohn disease,MONDO_0005011,CHEMBL4297477,BREPOCITINIB,Small molecule,PHASE_2
3,Crohn disease,MONDO_0005011,CHEMBL2104987,TEDUGLUTIDE,Protein,PHASE_2
4,Crohn disease,MONDO_0005011,CHEMBL1201821,PALIFERMIN,Protein,PHASE_1_2


## ClinicalTrials.gov: studies

In [4]:
trial_rows = []

for path in sorted(DATA_RAW_DIR.glob("clinical_trials_*.json")):
    query_bucket = path.stem.removeprefix("clinical_trials_")
    studies = json.loads(path.read_text())["studies"]

    for study in studies:
        protocol = study["protocolSection"]
        trial_rows.append({
            "query_bucket": query_bucket,
            "nct_id": protocol["identificationModule"].get("nctId"),
            "conditions": protocol.get("conditionsModule", {}).get("conditions", []),
            "keywords": protocol.get("conditionsModule", {}).get("keywords", []),
            "brief_summary": protocol.get("descriptionModule", {}).get("briefSummary"),
            "phases": protocol.get("designModule", {}).get("phases", []),
            "overall_status": protocol.get("statusModule", {}).get("overallStatus"),
            "lead_sponsor": protocol.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {}).get("name"),
        })

trials_df = pd.DataFrame(trial_rows)
trials_df.head()

,query_bucket,nct_id,conditions,keywords,brief_summary,phases,overall_status,lead_sponsor
0,crohns_disease,NCT02350920,[Inflammatory Bowel Disease],"[IBD, Crohn's disease, ulcerative colitis, ACT]","Over 18,000 Irish people are affected by the i...",[NA],COMPLETED,University College Dublin
1,crohns_disease,NCT03183661,[Crohn Disease],[],This is an open-label follow up study to evalu...,[],RECRUITING,"Anterogen Co., Ltd."
2,crohns_disease,NCT06424769,"[Inflammatory Bowel Diseases, Crohn Disease, U...","[Text Messaging, Symptom Monitoring, Social Ri...",The goal of this clinical trial is to learn wh...,[NA],ENROLLING_BY_INVITATION,"University of North Carolina, Chapel Hill"
3,crohns_disease,NCT01094613,[Crohn's Disease],"[Crohn's Disease, Delayed Release, Targeted Il...",The study is designed to evaluate the clinical...,"[PHASE1, PHASE2]",TERMINATED,Teva GTC
4,crohns_disease,NCT06734780,[The Impact of Vaping on Crohn's Disease Endos...,"[Crohn, e-cigarette, smoke, tobacco, heat-not-...",Tobacco smoke is a well-established risk facto...,[],UNKNOWN,IRCCS Ospedale San Raffaele
